<a href="https://colab.research.google.com/github/tschwider09/cross-species-ephys-transfer/blob/main/Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports

In [ ]:
# load human/mouse X (.npy) and y (.json)

import json
import numpy as np

# Here are example paths I used to export data from the feature extraction notebook you can replace with you own paths.
HUMAN_X_PATH = "human_X_seq.npy"
HUMAN_Y_PATH = "human_y_ttypes.json"
MOUSE_X_PATH = "mouse_X_seq.npy"
MOUSE_Y_PATH = "mouse_y_ttypes.json"

# Load X arrays
human_X_seq = np.load(HUMAN_X_PATH, allow_pickle=False)
mouse_X_seq = np.load(MOUSE_X_PATH, allow_pickle=False)

# Load y label lists from JSON
with open(HUMAN_Y_PATH, "r") as f:
    y_human = json.load(f)

with open(MOUSE_Y_PATH, "r") as f:
    mouse_y_ttypes = json.load(f)

# Basic sanity checks
print("human_X_seq:", human_X_seq.shape, human_X_seq.dtype)
print("mouse_X_seq:", mouse_X_seq.shape, mouse_X_seq.dtype)
print("human_y_ttypes:", type(y_human), "len =", len(y_human), "example =", y_human[:3])
print("mouse_y_ttypes:", type(mouse_y_ttypes), "len =", len(mouse_y_ttypes), "example =", mouse_y_ttypes[:3])

assert human_X_seq.shape[0] == len(y_human), "Human X/y length mismatch"
assert mouse_X_seq.shape[0] == len(mouse_y_ttypes), "Mouse X/y length mismatch"


In [ ]:
#Imports + sanity
import math, random

import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.patheffects as pe


from imblearn.over_sampling import SMOTE, RandomOverSampler


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

print("human_X_seq:", getattr(human_X_seq, "shape", None))
print("y_human:", getattr(y_human, "shape", None), type(y_human[0]) if len(y_human) else None)
print("mouse_X_seq:", getattr(mouse_X_seq, "shape", None))
print("mouse_y_ttypes:", getattr(mouse_y_ttypes, "shape", None), type(mouse_y_ttypes[0]) if len(mouse_y_ttypes) else None)


## Functions and Model Classes

In [ ]:
# Seeds + helpers

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def standardize_train_only(X_tr, X_va, X_te, eps=1e-6):
    mean = X_tr.mean(axis=0, keepdims=True)
    std  = X_tr.std(axis=0, keepdims=True)
    std  = np.where(std < eps, 1.0, std)
    return (X_tr-mean)/std, (X_va-mean)/std, (X_te-mean)/std, mean, std

#Smote Application

def smote_train(X_tr, y_tr, seed):

    N, T, D = X_tr.shape
    Xf = X_tr.reshape(N, T*D)
    min_count = min(Counter(y_tr.tolist()).values())

    if min_count >= 2:
        k = max(1, min(5, min_count - 1))
        sm = SMOTE(random_state=seed, k_neighbors=k)
        Xb, yb = sm.fit_resample(Xf, y_tr)
        return Xb.reshape(-1, T, D), yb, f"SMOTE(k={k})"

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.y[i]

@torch.no_grad()
def eval_logits_to_acc_f1(logits_list, y_list):
    y = np.concatenate(y_list, axis=0)
    p = np.concatenate([L.argmax(1) for L in logits_list], axis=0)
    return float(accuracy_score(y, p)), float(f1_score(y, p, average="macro"))

def inverse_freq_weights(y_int: np.ndarray, num_classes: int, device=device) -> torch.Tensor:
    cnt = Counter(y_int.tolist())
    w = torch.ones(num_classes, dtype=torch.float32)
    for c, k in cnt.items():
        w[c] = 1.0 / max(1, k)
    w = w * (len(cnt) / (w.sum() + 1e-12))
    return w.to(device)

def val_frac_inner_for_kfold(K_FOLDS, target_val=0.20):
    # inner val fraction on trainval pool so overall val ~= target_val when test=1/K
    return target_val / (1.0 - 1.0 / K_FOLDS)

# Feature-family names — must match the timestep order in X_seq[:, t, :]
FAMILY_NAMES = [
    "first_ap_v", "first_ap_dv", "isi_shape", "inst_freq",
    "spiking_threshold_v", "spiking_peak_v", "spiking_width",
    "spiking_fast_trough_v", "spiking_upstroke_downstroke_ratio",
    "step_subthresh", "subthresh_norm", "psth",
]

SHORT_FAMILY = {
    "first_ap_v": "1st AP V",
    "first_ap_dv": "1st AP dV/dt",
    "isi_shape": "ISI shape",
    "inst_freq": "Inst. freq",
    "spiking_threshold_v": "Thresh V",
    "spiking_peak_v": "Peak V",
    "spiking_width": "Width",
    "spiking_fast_trough_v": "Fast trough",
    "spiking_upstroke_downstroke_ratio": "Up/Down",
    "step_subthresh": "Subthresh step",
    "subthresh_norm": "Subthresh norm",
    "psth": "PSTH",
}


@torch.no_grad()
def collect_attn(trunk, loader, num_classes):
    """Aggregate per-class mean attention weights over a DataLoader."""
    trunk.eval()
    sums, counts = None, np.zeros(num_classes, dtype=np.int64)
    for xb, yb in loader:
        _, w = trunk(xb.to(device), return_emb=True, return_attn=True)
        w = w.cpu().numpy(); y = yb.numpy()
        if sums is None:
            sums = np.zeros((num_classes, w.shape[1]), dtype=np.float64)
        for c in range(num_classes):
            m = y == c
            if m.any():
                sums[c] += w[m].sum(0)
                counts[c] += m.sum()
    means = np.where(counts[:, None] > 0, sums / counts[:, None], np.nan)
    return means, counts


def attn_to_df(means, counts, class_names, family_names=FAMILY_NAMES):
    """Wrap mean attention + counts into a tidy DataFrame."""
    df = pd.DataFrame(means, index=class_names, columns=family_names)
    df.insert(0, "n_samples", counts)
    return df.round(3)


def plot_attn_heatmap(df, title, family_names=FAMILY_NAMES,
                      row_normalize=True, cmap="viridis",
                      figsize=(11.2, 3.8), annotate=True,
                      decimals=2, font_size=11, font_weight="bold",
                      outline_width=2.4, save_path=None):
    """Paper-ready attention heatmap from an attn_to_df DataFrame."""
    classes = list(df.index)
    mat     = df.loc[:, family_names].to_numpy(dtype=float)
    mat     = np.nan_to_num(mat, nan=0.0)

    if row_normalize:
        mat = mat / np.maximum(mat.sum(axis=1, keepdims=True), 1e-12)

    xlabels   = [SHORT_FAMILY.get(f, f) for f in family_names]
    cbar_label = "Mean attention (row-normalized)" if row_normalize else "Mean attention"

    fig, ax = plt.subplots(figsize=figsize)
    im   = ax.imshow(mat, aspect="auto", cmap=cmap, interpolation="nearest")
    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label(cbar_label, rotation=90)

    ax.set_yticks(np.arange(len(classes)));   ax.set_yticklabels(classes)
    ax.set_xticks(np.arange(len(xlabels)));   ax.set_xticklabels(xlabels, rotation=35, ha="right")
    ax.set_title(title)
    ax.set_xlabel("Electrophysiological feature family")
    ax.set_ylabel("Subclass")

    # subtle cell gridlines
    ax.set_xticks(np.arange(-.5, len(xlabels), 1), minor=True)
    ax.set_yticks(np.arange(-.5, len(classes),  1), minor=True)
    ax.grid(which="minor", linestyle="-", linewidth=0.6, alpha=0.25)
    ax.tick_params(which="minor", bottom=False, left=False)

    if annotate:
        vmin, vmax = mat.min(), mat.max()
        thresh = vmin + 0.55 * (vmax - vmin)
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                val = float(mat[i, j])
                txt_color = "white" if val < thresh else "black"
                outline   = "black" if txt_color == "white" else "white"
                t = ax.text(j, i, f"{val:.{decimals}f}",
                            ha="center", va="center",
                            fontsize=font_size, fontweight=font_weight,
                            color=txt_color)
                t.set_path_effects([pe.withStroke(linewidth=outline_width, foreground=outline)])

    plt.tight_layout()
    plt.show()

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"Saved: {save_path}")


In [ ]:
# Core models

class BiLSTMAttentionClassifier(nn.Module):
    def __init__(self, input_dim, num_classes,
                 hidden=128, layers=1, bidir=True,
                 dropout=0.1, attn_hidden=128, head_dropout=0.2):
        super().__init__()
        self.in_norm = nn.LayerNorm(input_dim)
        self.lstm = nn.LSTM(
            input_size=input_dim, hidden_size=hidden, num_layers=layers,
            batch_first=True, bidirectional=bidir,
            dropout=dropout if layers > 1 else 0.0
        )
        self.enc_dim = hidden * (2 if bidir else 1)
        self.attn_mlp = nn.Sequential(nn.Linear(self.enc_dim, attn_hidden), nn.Tanh())
        self.attn_vec = nn.Linear(attn_hidden, 1, bias=False)
        self.head = nn.Sequential(
            nn.LayerNorm(self.enc_dim),
            nn.Dropout(head_dropout),
            nn.Linear(self.enc_dim, num_classes)
        )

    def forward(self, x, return_emb=False, return_attn=False):
        x = self.in_norm(x)
        out, _ = self.lstm(x)                       # (B, T, enc_dim)
        u = self.attn_mlp(out)                      # (B, T, attn_hidden)
        scores = self.attn_vec(u).squeeze(-1)       # (B, T)
        w = torch.softmax(scores, dim=1)            # (B, T)
        pooled = torch.bmm(w.unsqueeze(1), out).squeeze(1)  # (B, enc_dim)
        if return_emb:
            return (pooled, w) if return_attn else pooled
        return (self.head(pooled), w) if return_attn else self.head(pooled)


class BiLSTMNoAttnClassifier(nn.Module):
    def __init__(self, input_dim, num_classes,
                 hidden=128, layers=1, bidir=True,
                 dropout=0.1, head_dropout=0.2):
        super().__init__()
        self.in_norm = nn.LayerNorm(input_dim)
        self.lstm = nn.LSTM(
            input_size=input_dim, hidden_size=hidden, num_layers=layers,
            batch_first=True, bidirectional=bidir,
            dropout=dropout if layers > 1 else 0.0
        )
        self.enc_dim = hidden * (2 if bidir else 1)
        self.head = nn.Sequential(
            nn.LayerNorm(self.enc_dim),
            nn.Dropout(head_dropout),
            nn.Linear(self.enc_dim, num_classes)
        )

    def forward(self, x):
        x = self.in_norm(x)
        out, _ = self.lstm(x)
        pooled = out[:, -1, :]  # last-timestep pooling
        return self.head(pooled)


## Mouse Models

In [ ]:
# Mouse Hyperparameters

X_seq = mouse_X_seq.astype(np.float32)
y_str = np.array(mouse_y_ttypes, dtype=object)

# Encode
le_single = LabelEncoder()
y_all = le_single.fit_transform(y_str.astype(str)).astype(np.int64)
num_classes = len(le_single.classes_)
print("Single-dataset classes:", list(le_single.classes_))

# Config
N_RUNS = 10
EPOCHS = 50
PATIENCE = 7
BATCH_TRAIN = 64
BATCH_EVAL = 256

HIDDEN=128; LAYERS=1; BIDIR=True; DROPOUT=0.1; ATTN_HIDDEN=128; HEAD_DROPOUT=0.2
LR=1e-3; WD=1e-4; GRAD_CLIP=2.0

In [ ]:



def train_val_test_once_single(run_idx, mode="nosmote_attn", base_seed=23):
    """
    mode in {"nosmote_noattn", "nosmote_attn", "smote_attn"}
    """
    seed = base_seed + run_idx
    set_seed(seed)

    # split
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X_seq, y_all, test_size=0.40, random_state=seed, stratify=y_all
    )
    X_va, X_te, y_va, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=seed, stratify=y_tmp
    )

    # standardize (train only)
    X_tr, X_va, X_te, _, _ = standardize_train_only(X_tr, X_va, X_te)

    used = "None"
    if mode == "smote_attn":
        X_tr, y_tr, used = smote_train(X_tr, y_tr, seed)

    # loaders
    tr_loader = DataLoader(SeqDataset(X_tr, y_tr), batch_size=BATCH_TRAIN, shuffle=True)
    va_loader = DataLoader(SeqDataset(X_va, y_va), batch_size=BATCH_EVAL, shuffle=False)
    te_loader = DataLoader(SeqDataset(X_te, y_te), batch_size=BATCH_EVAL, shuffle=False)

    # model
    D = X_tr.shape[-1]
    if mode == "nosmote_noattn":
        model = BiLSTMNoAttnClassifier(D, num_classes, hidden=HIDDEN, layers=LAYERS, bidir=BIDIR,
                                       dropout=DROPOUT, head_dropout=HEAD_DROPOUT).to(device)
    else:
        model = BiLSTMAttentionClassifier(D, num_classes, hidden=HIDDEN, layers=LAYERS, bidir=BIDIR,
                                          dropout=DROPOUT, attn_hidden=ATTN_HIDDEN, head_dropout=HEAD_DROPOUT).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    crit = nn.CrossEntropyLoss()

    def run_epoch(loader, train=True):
        model.train(train)
        losses, all_y, all_p = [], [], []
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = crit(logits, yb)
            if train:
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                opt.step()
            losses.append(float(loss.item()))
            all_y.extend(yb.detach().cpu().numpy())
            all_p.extend(logits.argmax(1).detach().cpu().numpy())
        return float(np.mean(losses)), float(accuracy_score(all_y, all_p))

    # early stop on VAL acc
    best_va, best_state, bad = -1.0, None, 0
    tag = mode.upper()
    for ep in range(1, EPOCHS+1):
        tr_loss, tr_acc = run_epoch(tr_loader, True)
        va_loss, va_acc = run_epoch(va_loader, False)
        print(f"[{tag} Run {run_idx+1}] Ep {ep:02d} | train {tr_loss:.3f}/{tr_acc:.3f} | val {va_loss:.3f}/{va_acc:.3f}")
        if va_acc > best_va:
            best_va, bad = va_acc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"[{tag} Run {run_idx+1}] Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state, strict=True)

    @torch.no_grad()
    def eval_loader(loader):
        model.eval()
        all_y, all_p = [], []
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            all_y.extend(yb.detach().cpu().numpy())
            all_p.extend(logits.argmax(1).detach().cpu().numpy())
        acc = float(accuracy_score(all_y, all_p))
        f1m = float(f1_score(all_y, all_p, average="macro"))
        return acc, f1m

    te_acc, te_f1 = eval_loader(te_loader)
    print(f"[{tag} Run {run_idx+1}] Method={used} | TEST acc={te_acc:.4f} F1={te_f1:.4f}\n")
    return te_acc, te_f1

def run_single_suite():
    modes = ["nosmote_noattn", "nosmote_attn", "smote_attn"]
    rows = []
    for mode in modes:
        accs, f1s = [], []
        for r in range(N_RUNS):
            a, f = train_val_test_once_single(r, mode=mode, base_seed=23)
            accs.append(a); f1s.append(f)
        rows.append({
            "mode": mode,
            "acc_mean": float(np.mean(accs)), "acc_sd": float(np.std(accs, ddof=1)),
            "f1_mean": float(np.mean(f1s)),   "f1_sd": float(np.std(f1s, ddof=1)),
        })
    return pd.DataFrame(rows)

single_results = run_single_suite()
single_results


In [ ]:
# ArcFace (upweighting embedded class groups) + CB-Focal (using higher lr for harder examples) + NAT prior correction (adjusting for expected distributions) (single-dataset)
alpha = .7 #nat prior multiplier

class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.35):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, emb, labels=None):
        W = F.normalize(self.weight, dim=1)
        x = F.normalize(emb, dim=1)
        cosine = F.linear(x, W)  # (B, C)

        if labels is None:
            return self.s * cosine

        sine = torch.sqrt(torch.clamp(1.0 - cosine**2, min=1e-9))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        one_hot = F.one_hot(labels, num_classes=cosine.size(1)).float().to(cosine.device)
        logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return logits * self.s

def class_balanced_focal_loss(logits, targets, class_counts, beta=0.999, gamma=2.0):
    n = torch.tensor(class_counts, dtype=torch.float32, device=logits.device)
    C = logits.size(1)
    eff_num = 1.0 - torch.pow(torch.tensor(beta, device=logits.device), n)
    cb_w = (1.0 - beta) / torch.clamp(eff_num, min=1e-12)
    cb_w = cb_w / (cb_w.sum() / C)
    w_samp = cb_w[targets]
    ce = F.cross_entropy(logits, targets, reduction="none")
    pt = torch.exp(-ce)
    fl = ((1 - pt) ** gamma) * ce
    return (w_samp * fl).mean()
@torch.no_grad()
def eval_test(loader, trunk, arc, log_priors):
      trunk.eval(); arc.eval()
      all_y, all_p = [], []
      for xb, yb in loader:
          xb, yb = xb.to(device), yb.to(device)
          emb = trunk(xb, return_emb=True)
          logits = arc(emb, None)
          logits = logits + alpha * log_priors  # NAT prior correction
          all_y.extend(yb.detach().cpu().numpy())
          all_p.extend(logits.argmax(1).detach().cpu().numpy())
      acc = float(accuracy_score(all_y, all_p))
      f1m = float(f1_score(all_y, all_p, average="macro"))
      return acc, f1m
def train_val_test_once_arc(run_idx, base_seed=23, arc_s=30.0, arc_m=0.35, cb_beta=0.999, gamma=2.0):
    seed = base_seed + run_idx
    set_seed(seed)

    # split
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X_seq.astype(np.float32), y_all, test_size=0.40, random_state=seed, stratify=y_all
    )
    X_va, X_te, y_va, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=seed, stratify=y_tmp
    )

    # NAT priors from PRE-balance train
    nat_counts = Counter(y_tr.tolist())
    priors = np.array([nat_counts.get(c, 1e-6) for c in range(num_classes)], dtype=np.float32)
    priors = priors / priors.sum()
    log_priors = torch.log(torch.tensor(priors, device=device))

    # standardize
    X_tr, X_va, X_te, _, _ = standardize_train_only(X_tr, X_va, X_te)

    # SMOTE/ROS train only
    X_tr_bal, y_tr_bal, used = smote_train(X_tr, y_tr, seed)

    # loaders
    tr_loader = DataLoader(SeqDataset(X_tr_bal, y_tr_bal), batch_size=BATCH_TRAIN, shuffle=True)
    va_loader = DataLoader(SeqDataset(X_va, y_va), batch_size=BATCH_EVAL, shuffle=False)
    te_loader = DataLoader(SeqDataset(X_te, y_te), batch_size=BATCH_EVAL, shuffle=False)

    # trunk = attention model (return_emb=True)
    D = X_tr.shape[-1]
    trunk = BiLSTMAttentionClassifier(D, num_classes, hidden=HIDDEN, layers=LAYERS, bidir=BIDIR,
                                      dropout=DROPOUT, attn_hidden=ATTN_HIDDEN, head_dropout=HEAD_DROPOUT).to(device)

    # infer emb dim
    xb0, _ = next(iter(va_loader))
    emb_dim = trunk(xb0.to(device), return_emb=True).shape[1]

    arc = ArcMarginProduct(emb_dim, num_classes, s=arc_s, m=arc_m).to(device)

    bal_counts = Counter(y_tr_bal.tolist())
    class_counts_bal = [bal_counts.get(c, 0) for c in range(num_classes)]

    params = list(trunk.parameters()) + list(arc.parameters())
    opt = torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-4)

    def run_epoch(loader, train=True):
        trunk.train(train); arc.train(train)
        losses, all_y, all_p = [], [], []
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            emb = trunk(xb, return_emb=True)
            logits = arc(emb, yb if train else None)
            #logits = logits + log_priors

            if train:
                loss = class_balanced_focal_loss(logits, yb, class_counts_bal, beta=cb_beta, gamma=gamma)
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(params, GRAD_CLIP)
                opt.step()
            else:
                loss = F.cross_entropy(logits, yb)

            losses.append(float(loss.item()))
            all_y.extend(yb.detach().cpu().numpy())
            all_p.extend(logits.argmax(1).detach().cpu().numpy())

        return float(np.mean(losses)), float(accuracy_score(all_y, all_p))

    # ES on VAL acc
    best_va, best_state, bad = -1.0, None, 0
    for ep in range(1, EPOCHS+1):
        tr_loss, tr_acc = run_epoch(tr_loader, True)
        va_loss, va_acc = run_epoch(va_loader, False)
        print(f"[ARC Run {run_idx+1}] Ep {ep:02d} | train {tr_loss:.3f}/{tr_acc:.3f} | val {va_loss:.3f}/{va_acc:.3f}")
        if va_acc > best_va:
            best_va, bad = va_acc, 0
            best_state = {
                "trunk": {k: v.detach().cpu().clone() for k, v in trunk.state_dict().items()},
                "arc":   {k: v.detach().cpu().clone() for k, v in arc.state_dict().items()},
            }
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"[ARC Run {run_idx+1}] Early stopping.")
                break

    if best_state is not None:
        trunk.load_state_dict(best_state["trunk"], strict=True)
        arc.load_state_dict(best_state["arc"], strict=True)
    te_acc, te_f1 = eval_test(te_loader,trunk, arc, log_priors)
    print(f"[ARC Run {run_idx+1}] Method={used} | TEST acc={te_acc:.4f} F1={te_f1:.4f}\n")
    return te_acc, te_f1, trunk, te_loader

# Multi-run
arc_accs, arc_f1s = [], []
for r in range(N_RUNS):
    a, f,mouse_trunk, mouse_te_loader = train_val_test_once_arc(r, base_seed=23)
    arc_accs.append(a); arc_f1s.append(f)

print("ARC Mean±SD acc:", float(np.mean(arc_accs)), float(np.std(arc_accs, ddof=1)))
print("ARC Mean±SD f1 :", float(np.mean(arc_f1s)), float(np.std(arc_f1s, ddof=1)))



## Human Models

In [ ]:
# Coarsening + aligned encoding

COARSE_CLASSES = ['LAMP5/PAX6/Other', 'PVALB', 'SST', 'VIP']

def normalize_to_coarse(label: str) -> str:
    s = str(label)
    if s in COARSE_CLASSES:
        return s
    up = s.upper()
    if "PVALB" in up: return "PVALB"
    if "SST"   in up: return "SST"
    if "VIP"   in up: return "VIP"
    if ("LAMP5" in up) or ("SNGC" in up): return "LAMP5/PAX6/Other"
    return "LAMP5/PAX6/Other"

def filter_and_coarsen(X: np.ndarray, y_str: np.ndarray, name: str):
    y_coarse = np.array([normalize_to_coarse(v) for v in y_str], dtype=object)
    keep = np.array([v in COARSE_CLASSES for v in y_coarse], dtype=bool)
    X2 = X[keep].astype(np.float32)
    y2 = y_coarse[keep].astype(object)
    print(f"{name}: kept {keep.sum()}/{len(keep)}")
    return X2, y2

human_X, human_yc = filter_and_coarsen(human_X_seq, np.array(y_human, dtype=object), "HUMAN")
mouse_X, mouse_yc = filter_and_coarsen(mouse_X_seq, np.array(mouse_y_ttypes, dtype=object), "MOUSE")

le = LabelEncoder()
le.fit(np.concatenate([human_yc, mouse_yc]).astype(str))
print("Aligned classes:", list(le.classes_))

human_y_int = le.transform(human_yc.astype(str)).astype(np.int64)
mouse_y_int = le.transform(mouse_yc.astype(str)).astype(np.int64)

num_classes_coarse = len(le.classes_)
assert num_classes_coarse == 4, f"Expected 4 coarse classes, got {num_classes_coarse}"


In [ ]:
# Config
N_RUNS  = 10
K_FOLDS = 5
BASE_SEED = 9

EPOCHS = 50
PATIENCE = 7
BATCH_TRAIN = 64
BATCH_EVAL  = 256

LR = 1e-3
WD = 1e-4
GRAD_CLIP = 2.0

EARLY_STOP_ON = "f1"  # "f1" is usually better than "acc" under imbalance

In [ ]:
# HUMAN K-fold baselines (attn; with/without SMOTE)

def run_human_fold_baseline(train_idx, val_idx, test_idx, fold_seed, use_smote: bool):
    set_seed(fold_seed)

    X = human_X.astype(np.float32)
    y = human_y_int

    X_tr, y_tr = X[train_idx], y[train_idx]
    X_va, y_va = X[val_idx],   y[val_idx]
    X_te, y_te = X[test_idx],  y[test_idx]

    # standardize
    X_tr, X_va, X_te, _, _ = standardize_train_only(X_tr, X_va, X_te)

    used = "None"
    if use_smote:
        X_tr, y_tr, used = _maybe_smote_train_only(X_tr, y_tr, fold_seed)

    tr_loader = DataLoader(SeqDataset(X_tr, y_tr), batch_size=BATCH_TRAIN, shuffle=True)
    va_loader = DataLoader(SeqDataset(X_va, y_va), batch_size=BATCH_EVAL,  shuffle=False)
    te_loader = DataLoader(SeqDataset(X_te, y_te), batch_size=BATCH_EVAL,  shuffle=False)

    D = X_tr.shape[-1]
    model = BiLSTMAttentionClassifier(D, num_classes_coarse).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    # (optional) class weights when NOT using SMOTE (often helps)
    w = None
    if not use_smote:
        w = inverse_freq_weights(y_tr, num_classes_coarse).to(device)
    crit = nn.CrossEntropyLoss(weight=w)

    def epoch(loader, train=True):
        model.train(train)
        losses, all_y, all_p = [], [], []
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = crit(logits, yb)
            if train:
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                opt.step()
            losses.append(float(loss.item()))
            all_y.extend(yb.detach().cpu().numpy())
            all_p.extend(logits.argmax(1).detach().cpu().numpy())
        acc = float(accuracy_score(all_y, all_p))
        f1m = float(f1_score(all_y, all_p, average="macro"))
        return float(np.mean(losses)), acc, f1m

    best_metric, best_state, bad = -1.0, None, 0
    for ep in range(1, EPOCHS+1):
        tr_loss, tr_acc, tr_f1 = epoch(tr_loader, True)
        va_loss, va_acc, va_f1 = epoch(va_loader, False)
        metric = va_f1 if EARLY_STOP_ON == "f1" else va_acc
        if metric > best_metric:
            best_metric, bad = metric, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state, strict=True)

    # test
    model.eval()
    all_y, all_p = [], []
    with torch.no_grad():
        for xb, yb in te_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            all_y.extend(yb.detach().cpu().numpy())
            all_p.extend(logits.argmax(1).detach().cpu().numpy())
    te_acc = float(accuracy_score(all_y, all_p))
    te_f1  = float(f1_score(all_y, all_p, average="macro"))
    return te_acc, te_f1, used

def run_human_kfold_regime(use_smote: bool):
    tag = "HUMAN_Attn_SMOTE" if use_smote else "HUMAN_Attn_noSMOTE"
    vf = val_frac_inner_for_kfold(K_FOLDS, target_val=0.20)

    run_accs, run_f1s = [], []
    for run in range(N_RUNS):
        seed = BASE_SEED + run
        set_seed(seed)

        skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=seed)
        fold_accs, fold_f1s = [], []

        for fold_i, (trainval_idx, test_idx) in enumerate(skf.split(human_X, human_y_int), start=1):
            fold_seed = seed + 1000 * fold_i
            tr_idx, va_idx = train_test_split(
                np.array(trainval_idx, int),
                test_size=vf,
                random_state=fold_seed,
                stratify=human_y_int[np.array(trainval_idx, int)]
            )
            acc, f1m, used = run_human_fold_baseline(np.array(tr_idx,int), np.array(va_idx,int), np.array(test_idx,int),
                                                     fold_seed, use_smote)
            fold_accs.append(acc); fold_f1s.append(f1m)

        run_accs.append(float(np.mean(fold_accs)))
        run_f1s.append(float(np.mean(fold_f1s)))

    return {
        "tag": tag,
        "acc_mean": float(np.mean(run_accs)), "acc_sd": float(np.std(run_accs, ddof=1)),
        "f1_mean": float(np.mean(run_f1s)),   "f1_sd": float(np.std(run_f1s, ddof=1)),
    }

human_kfold_results = pd.DataFrame([
    run_human_kfold_regime(use_smote=True),
    run_human_kfold_regime(use_smote=False),
])
human_kfold_results


In [ ]:
#HUMAN ArcFace K-fold

ARC_S = 30.0
ARC_M = 0.35
CB_BETA = 0.999
FOCAL_GAMMA = 2.0

def run_human_fold_arcface(train_idx, val_idx, test_idx, fold_seed):
    set_seed(fold_seed)

    X = human_X.astype(np.float32)
    y = human_y_int

    X_tr, y_tr = X[train_idx], y[train_idx]
    X_va, y_va = X[val_idx],   y[val_idx]
    X_te, y_te = X[test_idx],  y[test_idx]

    # NAT priors from PRE-SMOTE train
    nat_counts = Counter(y_tr.tolist())
    priors = np.array([nat_counts.get(c, 1e-6) for c in range(num_classes_coarse)], dtype=np.float32)
    priors = priors / priors.sum()
    log_priors = torch.log(torch.tensor(priors, device=device))

    # standardize
    X_tr, X_va, X_te, _, _ = standardize_train_only(X_tr, X_va, X_te)

    # SMOTE/ROS on TRAIN only
    X_tr_bal, y_tr_bal, used = smote_train(X_tr, y_tr, fold_seed)

    # loaders
    tr_loader = DataLoader(SeqDataset(X_tr_bal, y_tr_bal), batch_size=BATCH_TRAIN, shuffle=True)
    va_loader = DataLoader(SeqDataset(X_va, y_va), batch_size=BATCH_EVAL, shuffle=False)
    te_loader = DataLoader(SeqDataset(X_te, y_te), batch_size=BATCH_EVAL, shuffle=False)

    # trunk
    D = X_tr.shape[-1]
    trunk = BiLSTMAttentionClassifier(D, num_classes_coarse).to(device)
    xb0, _ = next(iter(va_loader))
    emb_dim = trunk(xb0.to(device), return_emb=True).shape[1]
    arc = ArcMarginProduct(emb_dim, num_classes_coarse, s=ARC_S, m=ARC_M).to(device)

    bal_counts = Counter(y_tr_bal.tolist())
    class_counts_bal = [bal_counts.get(c, 0) for c in range(num_classes_coarse)]

    params = list(trunk.parameters()) + list(arc.parameters())
    opt = torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-4)

    def epoch(loader, train=True):
        trunk.train(train); arc.train(train)
        losses, all_y, all_p = [], [], []
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            emb = trunk(xb, return_emb=True)
            logits = arc(emb, yb if train else None)
            if train:
                loss = class_balanced_focal_loss(logits, yb, class_counts_bal, beta=CB_BETA, gamma=FOCAL_GAMMA)
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(params, GRAD_CLIP)
                opt.step()
            else:
                loss = F.cross_entropy(logits, yb)
            losses.append(float(loss.item()))
            all_y.extend(yb.detach().cpu().numpy())
            all_p.extend(logits.argmax(1).detach().cpu().numpy())
        acc = float(accuracy_score(all_y, all_p))
        f1m = float(f1_score(all_y, all_p, average="macro"))
        return float(np.mean(losses)), acc, f1m

    # ES on VAL accuracy (to match your earlier ARC script)
    best_va_acc, best_state, bad = -1.0, None, 0
    for ep in range(1, EPOCHS+1):
        tr_loss, tr_acc, tr_f1 = epoch(tr_loader, True)
        va_loss, va_acc, va_f1 = epoch(va_loader, False)
        if va_acc > best_va_acc:
            best_va_acc, bad = va_acc, 0
            best_state = {
                "trunk": {k: v.detach().cpu().clone() for k, v in trunk.state_dict().items()},
                "arc":   {k: v.detach().cpu().clone() for k, v in arc.state_dict().items()},
            }
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    if best_state is not None:
        trunk.load_state_dict(best_state["trunk"], strict=True)
        arc.load_state_dict(best_state["arc"], strict=True)

    # TEST + NAT prior correction
    trunk.eval(); arc.eval()
    all_y, all_p = [], []
    with torch.no_grad():
        for xb, yb in te_loader:
            xb, yb = xb.to(device), yb.to(device)
            emb = trunk(xb, return_emb=True)
            logits = arc(emb, None)
            logits = logits - log_priors
            all_y.extend(yb.detach().cpu().numpy())
            all_p.extend(logits.argmax(1).detach().cpu().numpy())
    human_trunk = trunk       # For agg attention
    human_te_loader = te_loader
    te_acc = float(accuracy_score(all_y, all_p))
    te_f1  = float(f1_score(all_y, all_p, average="macro"))
    return te_acc, te_f1, used, trunk, te_loader

def run_human_kfold_arcface():
    vf = val_frac_inner_for_kfold(K_FOLDS, target_val=0.20)
    run_accs, run_f1s = [], []
    for run in range(N_RUNS):
        seed = BASE_SEED + run
        set_seed(seed)

        skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=seed)
        fold_accs, fold_f1s = [], []

        for fold_i, (trainval_idx, test_idx) in enumerate(skf.split(human_X, human_y_int), start=1):
            fold_seed = seed + 1000 * fold_i
            tr_idx, va_idx = train_test_split(
                np.array(trainval_idx, int),
                test_size=vf,
                random_state=fold_seed,
                stratify=human_y_int[np.array(trainval_idx, int)]
            )
            acc, f1m, used, trunk, te_loader = run_human_fold_arcface(np.array(tr_idx,int), np.array(va_idx,int), np.array(test_idx,int), fold_seed)
            fold_accs.append(acc); fold_f1s.append(f1m)

        run_accs.append(float(np.mean(fold_accs)))
        run_f1s.append(float(np.mean(fold_f1s)))

    return {
        "tag": "HUMAN_ArcFace_SMOTE_CBfocal_NAT",
        "acc_mean": float(np.mean(run_accs)), "acc_sd": float(np.std(run_accs, ddof=1)),
        "f1_mean": float(np.mean(run_f1s)),   "f1_sd": float(np.std(run_f1s, ddof=1)),
    }, trunk, te_loader

arc_kfold_summary, human_trunk, human_te_loader = run_human_kfold_arcface()


pd.DataFrame([arc_kfold_summary])

## Attention Diagrams

In [ ]:
mouse_attn_means, mouse_attn_counts = collect_attn(mouse_trunk, mouse_te_loader, num_classes)
mouse_attn_df = attn_to_df(mouse_attn_means, mouse_attn_counts, list(le_single.classes_))

print("=== Mouse ArcFace+SMOTE — mean attention by class (TEST) ===")
print(mouse_attn_df.to_string())


In [ ]:
plot_attn_heatmap(mouse_attn_df,
                  title="Mouse inhibitory subclasses: mean attention by feature family")

In [ ]:
human_attn_means, human_attn_counts = collect_attn(human_trunk, human_te_loader, num_classes_coarse)
human_attn_df = attn_to_df(human_attn_means, human_attn_counts, list(le.classes_))

print("=== Human ArcFace+SMOTE — mean attention by class (TEST) ===")
print(human_attn_df.to_string())

In [ ]:
plot_attn_heatmap(human_attn_df,
                  title="Human inhibitory subclasses: mean attention by feature family")

## Transfer Models

In [ ]:
# Normal transfer (K-fold): mouse pretrain once/run, then fine-tune each human fold

# Config
N_RUNS = 10
BASE_SEED = 36
K_FOLDS = 5
vf = val_frac_inner_for_kfold(K_FOLDS, target_val=0.20)

EPOCHS_BASE = 50; PATIENCE_BASE = 7; LR_BASE = 1e-3; WD_BASE = 1e-4
EPOCHS_MOUSE = 50; PATIENCE_MOUSE = 7; LR_MOUSE = 1e-3; WD_MOUSE = 1e-4
EPOCHS_FT = 50; PATIENCE_FT = 7; LR_FT = 5e-4; WD_FT = 1e-4

USE_WARMUP = True
WARMUP_EPOCHS = 5
LR_WARMUP = 1e-3
WD_WARMUP = 1e-4

TRANSFER_REINIT_HEAD = True  # "normal transfer": copy encoder only, new head

class BaselineModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.encoder = BiLSTMAttentionClassifier(input_dim, num_classes)  # we'll use its encoder path via return_emb
        # Replace its head with a fresh head so we can cleanly copy encoder only
        enc_dim = self.encoder.enc_dim
        self.head = nn.Sequential(nn.LayerNorm(enc_dim), nn.Dropout(0.2), nn.Linear(enc_dim, num_classes))
    def forward(self, x):
        z = self.encoder(x, return_emb=True)
        return self.head(z)

def copy_encoder(dst: BaselineModel, src: BaselineModel):
    # copy ONLY LSTM+attn parts (everything inside encoder except its head)
    dst_sd = dst.state_dict()
    src_sd = src.state_dict()
    for k in list(dst_sd.keys()):
        if k.startswith("encoder.") and (".head." not in k):
            if k in src_sd and src_sd[k].shape == dst_sd[k].shape:
                dst_sd[k] = src_sd[k]
    dst.load_state_dict(dst_sd, strict=False)

def train_epochs_es(model, tr_loader, va_loader, crit, opt, patience, early_on="acc", tag=""):
    best_metric, best_state, bad = -1.0, None, 0
    for ep in range(1, 10_000):
        # train
        model.train(True)
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = crit(model(xb), yb)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        # val
        model.eval()
        all_y, all_p = [], []
        with torch.no_grad():
            for xb, yb in va_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                all_y.extend(yb.detach().cpu().numpy())
                all_p.extend(logits.argmax(1).detach().cpu().numpy())
        va_acc = float(accuracy_score(all_y, all_p))
        va_f1  = float(f1_score(all_y, all_p, average="macro"))

        metric = va_acc if early_on == "acc" else va_f1
        if metric > best_metric:
            best_metric, bad = metric, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
        if ep >= EPOCHS_BASE and ("BASE" in tag):
            break
        if ep >= EPOCHS_MOUSE and ("MOUSE" in tag):
            break
        if ep >= EPOCHS_FT and ("FT" in tag):
            break
        if ep >= WARMUP_EPOCHS and ("WARM" in tag):
            break

    if best_state is not None:
        model.load_state_dict(best_state, strict=True)
    return model

@torch.no_grad()
def test_acc_f1(model, te_loader):
    model.eval()
    all_y, all_p = [], []
    for xb, yb in te_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        all_y.extend(yb.detach().cpu().numpy())
        all_p.extend(logits.argmax(1).detach().cpu().numpy())
    return float(accuracy_score(all_y, all_p)), float(f1_score(all_y, all_p, average="macro"))

def run_kfold_normal_transfer():
    run_base_accs, run_base_f1s = [], []
    run_tr_accs, run_tr_f1s = [], []
    run_dacc, run_df1 = [], []

    for run in range(N_RUNS):
        seed = BASE_SEED + run
        set_seed(seed)

        # ----- mouse pretrain split once per run (80/20)
        Xm_tr, Xm_va, ym_tr, ym_va = train_test_split(
            mouse_X.astype(np.float32), mouse_y_int,
            test_size=0.20, random_state=seed, stratify=mouse_y_int
        )
        Xm_tr, Xm_va, _, _, _ = standardize_train_only(Xm_tr, Xm_va, Xm_va.copy())
        m_tr = DataLoader(SeqDataset(Xm_tr, ym_tr), batch_size=BATCH_TRAIN, shuffle=True)
        m_va = DataLoader(SeqDataset(Xm_va, ym_va), batch_size=BATCH_EVAL,  shuffle=False)

        D = Xm_tr.shape[-1]
        mouse_model = BaselineModel(D, num_classes_coarse).to(device)
        w_m = inverse_freq_weights(ym_tr, num_classes_coarse).to(device)
        crit_m = nn.CrossEntropyLoss(weight=w_m)
        opt_m = torch.optim.AdamW(mouse_model.parameters(), lr=LR_MOUSE, weight_decay=WD_MOUSE)

        mouse_model = train_epochs_es(mouse_model, m_tr, m_va, crit_m, opt_m, PATIENCE_MOUSE, early_on="f1", tag="MOUSE")

        # ----- human folds
        skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=seed)
        fold_base_accs, fold_base_f1s = [], []
        fold_tr_accs, fold_tr_f1s = [], []

        for fold_i, (trainval_idx, test_idx) in enumerate(skf.split(human_X, human_y_int), start=1):
            fold_seed = seed + 1000 * fold_i
            set_seed(fold_seed)

            trainval_idx = np.array(trainval_idx, int)
            test_idx = np.array(test_idx, int)

            tr_idx, va_idx = train_test_split(
                trainval_idx, test_size=vf, random_state=fold_seed,
                stratify=human_y_int[trainval_idx]
            )
            tr_idx = np.array(tr_idx, int); va_idx = np.array(va_idx, int)

            Xh_tr, yh_tr = human_X[tr_idx].astype(np.float32), human_y_int[tr_idx]
            Xh_va, yh_va = human_X[va_idx].astype(np.float32), human_y_int[va_idx]
            Xh_te, yh_te = human_X[test_idx].astype(np.float32), human_y_int[test_idx]

            Xh_tr, Xh_va, Xh_te, _, _ = standardize_train_only(Xh_tr, Xh_va, Xh_te)

            h_tr = DataLoader(SeqDataset(Xh_tr, yh_tr), batch_size=BATCH_TRAIN, shuffle=True)
            h_va = DataLoader(SeqDataset(Xh_va, yh_va), batch_size=BATCH_EVAL,  shuffle=False)
            h_te = DataLoader(SeqDataset(Xh_te, yh_te), batch_size=BATCH_EVAL,  shuffle=False)

            w_h = inverse_freq_weights(yh_tr, num_classes_coarse).to(device)
            crit_h = nn.CrossEntropyLoss(weight=w_h)

            # ---- baseline (human scratch)
            base = BaselineModel(Xh_tr.shape[-1], num_classes_coarse).to(device)
            opt_b = torch.optim.AdamW(base.parameters(), lr=LR_BASE, weight_decay=WD_BASE)
            base = train_epochs_es(base, h_tr, h_va, crit_h, opt_b, PATIENCE_BASE, early_on="acc", tag="BASE")
            b_acc, b_f1 = test_acc_f1(base, h_te)

            # ---- transfer model
            set_seed(fold_seed)  # keep init comparable
            trm = BaselineModel(Xh_tr.shape[-1], num_classes_coarse).to(device)
            copy_encoder(trm, mouse_model)
            if (not TRANSFER_REINIT_HEAD):
                trm.head.load_state_dict(mouse_model.head.state_dict(), strict=True)

            # warmup head-only
            if USE_WARMUP:
                for p in trm.encoder.parameters():
                    p.requires_grad = False
                opt_w = torch.optim.AdamW(trm.parameters(), lr=LR_WARMUP, weight_decay=WD_WARMUP)
                trm = train_epochs_es(trm, h_tr, h_va, crit_h, opt_w, PATIENCE_FT, early_on="acc", tag="WARM")
                for p in trm.encoder.parameters():
                    p.requires_grad = True

            # full fine-tune
            opt_t = torch.optim.AdamW(trm.parameters(), lr=LR_FT, weight_decay=WD_FT)
            trm = train_epochs_es(trm, h_tr, h_va, crit_h, opt_t, PATIENCE_FT, early_on="acc", tag="FT")
            t_acc, t_f1 = test_acc_f1(trm, h_te)

            fold_base_accs.append(b_acc); fold_base_f1s.append(b_f1)
            fold_tr_accs.append(t_acc);   fold_tr_f1s.append(t_f1)

        base_acc_run = float(np.mean(fold_base_accs)); base_f1_run = float(np.mean(fold_base_f1s))
        tr_acc_run   = float(np.mean(fold_tr_accs));   tr_f1_run   = float(np.mean(fold_tr_f1s))

        run_base_accs.append(base_acc_run); run_base_f1s.append(base_f1_run)
        run_tr_accs.append(tr_acc_run);     run_tr_f1s.append(tr_f1_run)
        run_dacc.append(tr_acc_run - base_acc_run)
        run_df1.append(tr_f1_run - base_f1_run)

        print(f"[RUN {run+1}] BASE acc={base_acc_run:.4f} F1={base_f1_run:.4f} | "
              f"TR acc={tr_acc_run:.4f} F1={tr_f1_run:.4f} | Δacc={run_dacc[-1]:+.4f} ΔF1={run_df1[-1]:+.4f}")

    return {
        "BASE_acc_mean": float(np.mean(run_base_accs)), "BASE_acc_sd": float(np.std(run_base_accs, ddof=1)),
        "BASE_f1_mean":  float(np.mean(run_base_f1s)),  "BASE_f1_sd":  float(np.std(run_base_f1s, ddof=1)),
        "TR_acc_mean":   float(np.mean(run_tr_accs)),    "TR_acc_sd":   float(np.std(run_tr_accs, ddof=1)),
        "TR_f1_mean":    float(np.mean(run_tr_f1s)),     "TR_f1_sd":    float(np.std(run_tr_f1s, ddof=1)),
        "Dacc_mean":     float(np.mean(run_dacc)),       "Dacc_sd":     float(np.std(run_dacc, ddof=1)),
        "Df1_mean":      float(np.mean(run_df1)),        "Df1_sd":      float(np.std(run_df1, ddof=1)),
    }

normal_transfer_summary = run_kfold_normal_transfer()
normal_transfer_summary


In [ ]:
# (Joint MTL + human finetune) on SAME human folds as baseline
# Base it off your paired runner spec:
#  - Outer StratifiedKFold -> TEST
#  - Inner stratified split on remaining -> TRAIN/VAL (val_frac_inner = 0.20 / (1 - 1/K))
#  - Train-only standardization (per fold for human; per run for mouse train split)
#  - Baseline: class-weighted CE, AdamW, early stop on VAL ACC (checkpoint best VAL acc)
#  - joint on mouse_train + human_train:
#       L = L_human + alpha_mouse * L_mouse
#     early stop on HUMAN val macro-F1
#    then finetune on human_train only (same split), early stop on HUMAN val macro-F1

# Config
N_RUNS     = 10
BASE_SEED  = 36
K_FOLDS    = 5

EPOCHS_BASE  = 50;  PATIENCE_BASE  = 7;  LR_BASE  = 1e-3;  WD_BASE  = 1e-4
EPOCHS_JOINT = 50;  PATIENCE_JOINT = 7;  LR_JOINT = 5e-4;  WD_JOINT = 1e-4
EPOCHS_FT    = 50;  PATIENCE_FT    = 7;  LR_FT    = 3e-4;  WD_FT    = 1e-4

ALPHA_MOUSE = 0.5

# -----------------------------
# Split helpers
# -----------------------------

def stratified_split_indices(trainval_idx, y_int, val_frac, seed):
    tr, va = train_test_split(
        trainval_idx, test_size=val_frac,
        random_state=seed, stratify=y_int[trainval_idx]
    )
    return np.array(tr, dtype=int), np.array(va, dtype=int)


def stratified_split_80_20(X: np.ndarray, y_int: np.ndarray, seed: int):
    return train_test_split(X, y_int, test_size=0.20, random_state=seed, stratify=y_int)


# -----------------------------
# Model: Two-head MTL
# (shared BiLSTM+attention encoder, separate human and mouse heads)
# -----------------------------

class TwoHeadMTLModel(nn.Module):
    def __init__(self, input_dim, num_classes_human, num_classes_mouse):
        super().__init__()
        self._backbone = BiLSTMAttentionClassifier(input_dim, num_classes_human)
        enc_dim = self._backbone.enc_dim
        self.head_h = nn.Sequential(
            nn.LayerNorm(enc_dim), nn.Dropout(HEAD_DROPOUT), nn.Linear(enc_dim, num_classes_human)
        )
        self.head_m = nn.Sequential(
            nn.LayerNorm(enc_dim), nn.Dropout(HEAD_DROPOUT), nn.Linear(enc_dim, num_classes_mouse)
        )

    def forward_human(self, x):
        return self.head_h(self._backbone(x, return_emb=True))

    def forward_mouse(self, x):
        return self.head_m(self._backbone(x, return_emb=True))


# -----------------------------
# Baseline fold trainer (ES on VAL ACC)
# -----------------------------

def train_baseline_human_fold(
    run_idx, fold_idx,
    X_tr, y_tr, X_va, y_va, X_te, y_te,
    num_classes
):
    w    = inverse_freq_weights(y_tr, num_classes).to(device)
    crit = nn.CrossEntropyLoss(weight=w)

    train_loader = DataLoader(SeqDataset(X_tr, y_tr), batch_size=BATCH_TRAIN, shuffle=True)
    val_loader   = DataLoader(SeqDataset(X_va, y_va), batch_size=BATCH_EVAL,  shuffle=False)
    test_loader  = DataLoader(SeqDataset(X_te, y_te), batch_size=BATCH_EVAL,  shuffle=False)

    model = BiLSTMAttentionClassifier(input_dim=X_tr.shape[-1], num_classes=num_classes).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR_BASE, weight_decay=WD_BASE)

    best_va_acc = -1.0
    best_state  = None
    bad         = 0

    for ep in range(1, EPOCHS_BASE + 1):
        # train
        model.train(True)
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss   = crit(logits, yb)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        # val
        model.eval()
        all_y, all_p, val_losses = [], [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                val_losses.append(float(crit(logits, yb).item()))
                all_y.extend(yb.detach().cpu().numpy())
                all_p.extend(logits.argmax(1).detach().cpu().numpy())

        va_loss = float(np.mean(val_losses)) if val_losses else float("nan")
        va_acc  = float(accuracy_score(all_y, all_p))
        va_f1   = float(f1_score(all_y, all_p, average="macro"))
        print(f"[HUMAN-BASELINE Run {run_idx} Fold {fold_idx}] Ep {ep:02d} | "
              f"val {va_loss:.3f}/{va_acc:.3f}/F1={va_f1:.3f}")

        if va_acc > best_va_acc:
            best_va_acc = va_acc
            bad = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE_BASE:
                print(f"[HUMAN-BASELINE Run {run_idx} Fold {fold_idx}] Early stopping (VAL ACC).")
                break

    if best_state is not None:
        model.load_state_dict(best_state, strict=True)

    # test
    model.eval()
    all_y, all_p = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            all_y.extend(yb.detach().cpu().numpy())
            all_p.extend(model(xb).argmax(dim=1).detach().cpu().numpy())

    te_acc = float(accuracy_score(all_y, all_p))
    te_f1  = float(f1_score(all_y, all_p, average="macro"))
    return te_acc, te_f1


# -----------------------------
# Joint-then-finetune fold trainer: JOINT (ES on HUMAN val macro-F1) -> FINETUNE (ES on HUMAN val macro-F1)
# -----------------------------

def train_joint_finetune_fold(
    run_idx, fold_idx,
    X_h_tr, y_h_tr, X_h_va, y_h_va, X_h_te, y_h_te,
    X_m_tr, y_m_tr,
    num_classes_h, num_classes_m
):
    # Loaders
    h_train = DataLoader(SeqDataset(X_h_tr, y_h_tr), batch_size=BATCH_TRAIN, shuffle=True,  drop_last=True)
    h_val   = DataLoader(SeqDataset(X_h_va, y_h_va), batch_size=BATCH_EVAL,  shuffle=False)
    h_test  = DataLoader(SeqDataset(X_h_te, y_h_te), batch_size=BATCH_EVAL,  shuffle=False)
    m_train = DataLoader(SeqDataset(X_m_tr, y_m_tr), batch_size=BATCH_TRAIN, shuffle=True,  drop_last=True)

    # Loss weights (train only)
    crit_h = nn.CrossEntropyLoss(weight=inverse_freq_weights(y_h_tr, num_classes_h).to(device))
    crit_m = nn.CrossEntropyLoss(weight=inverse_freq_weights(y_m_tr, num_classes_m).to(device))

    model = TwoHeadMTLModel(
        input_dim=X_h_tr.shape[-1],
        num_classes_human=num_classes_h,
        num_classes_mouse=num_classes_m
    ).to(device)

    # ---------------- JOINT ----------------
    opt = torch.optim.AdamW(model.parameters(), lr=LR_JOINT, weight_decay=WD_JOINT)
    best_va_f1 = -1.0
    best_state = None
    bad        = 0

    for ep in range(1, EPOCHS_JOINT + 1):
        model.train(True)
        joint_losses = []

        h_iter = iter(h_train)
        m_iter = iter(m_train)
        steps  = max(len(h_train), len(m_train))

        for _ in range(steps):
            try:    xh, yh = next(h_iter)
            except StopIteration: h_iter = iter(h_train); xh, yh = next(h_iter)
            try:    xm, ym = next(m_iter)
            except StopIteration: m_iter = iter(m_train); xm, ym = next(m_iter)

            xh, yh = xh.to(device), yh.to(device)
            xm, ym = xm.to(device), ym.to(device)

            loss_h = crit_h(model.forward_human(xh), yh)
            loss_m = crit_m(model.forward_mouse(xm), ym)
            loss   = loss_h + (ALPHA_MOUSE * loss_m)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            joint_losses.append(float(loss.item()))

        # human val
        model.eval()
        val_losses, all_y, all_p = [], [], []
        with torch.no_grad():
            for xb, yb in h_val:
                xb, yb = xb.to(device), yb.to(device)
                logits = model.forward_human(xb)
                val_losses.append(float(crit_h(logits, yb).item()))
                all_y.extend(yb.detach().cpu().numpy())
                all_p.extend(logits.argmax(1).detach().cpu().numpy())

        va_loss = float(np.mean(val_losses)) if val_losses else float("nan")
        va_acc  = float(accuracy_score(all_y, all_p))
        va_f1   = float(f1_score(all_y, all_p, average="macro"))
        print(f"[JOINT-FT Run {run_idx} Fold {fold_idx}] Ep {ep:02d} | "
              f"joint_train_loss {float(np.mean(joint_losses)):.3f} | "
              f"human_val {va_loss:.3f}/{va_acc:.3f}/F1={va_f1:.3f}")

        if va_f1 > best_va_f1:
            best_va_f1 = va_f1
            bad = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE_JOINT:
                print(f"[JOINT-FT Run {run_idx} Fold {fold_idx}] Early stopping (HUMAN val macro-F1).")
                break

    if best_state is not None:
        model.load_state_dict(best_state, strict=True)

    # ---------------- FINETUNE (human only) ----------------
    opt_ft = torch.optim.AdamW(model.parameters(), lr=LR_FT, weight_decay=WD_FT)
    best_ft_f1    = -1.0
    best_ft_state = None
    bad           = 0

    for ep in range(1, EPOCHS_FT + 1):
        model.train(True)
        tr_losses = []
        for xb, yb in h_train:
            xb, yb = xb.to(device), yb.to(device)
            loss = crit_h(model.forward_human(xb), yb)
            opt_ft.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt_ft.step()
            tr_losses.append(float(loss.item()))

        # val
        model.eval()
        val_losses, all_y, all_p = [], [], []
        with torch.no_grad():
            for xb, yb in h_val:
                xb, yb = xb.to(device), yb.to(device)
                logits = model.forward_human(xb)
                val_losses.append(float(crit_h(logits, yb).item()))
                all_y.extend(yb.detach().cpu().numpy())
                all_p.extend(logits.argmax(1).detach().cpu().numpy())

        va_loss = float(np.mean(val_losses)) if val_losses else float("nan")
        va_acc  = float(accuracy_score(all_y, all_p))
        va_f1   = float(f1_score(all_y, all_p, average="macro"))
        print(f"[FINETUNE-HUMAN Run {run_idx} Fold {fold_idx}] Ep {ep:02d} | "
              f"train_loss {float(np.mean(tr_losses)):.3f} | "
              f"val {va_loss:.3f}/{va_acc:.3f}/F1={va_f1:.3f}")

        if va_f1 > best_ft_f1:
            best_ft_f1 = va_f1
            bad = 0
            best_ft_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE_FT:
                print(f"[FINETUNE-HUMAN Run {run_idx} Fold {fold_idx}] Early stopping (val macro-F1).")
                break

    if best_ft_state is not None:
        model.load_state_dict(best_ft_state, strict=True)

    # test (human)
    model.eval()
    all_y, all_p = [], []
    with torch.no_grad():
        for xb, yb in h_test:
            xb, yb = xb.to(device), yb.to(device)
            all_y.extend(yb.detach().cpu().numpy())
            all_p.extend(model.forward_human(xb).argmax(dim=1).detach().cpu().numpy())

    te_acc = float(accuracy_score(all_y, all_p))
    te_f1  = float(f1_score(all_y, all_p, average="macro"))
    return te_acc, te_f1


# -----------------------------
# K-FOLD paired driver
# -----------------------------

vf = val_frac_inner_for_kfold(K_FOLDS, target_val=0.20)

run_base_accs, run_base_f1s   = [], []
run_jf_accs,    run_jf_f1s      = [], []
run_delta_accs, run_delta_f1s = [], []

for run in range(N_RUNS):
    seed = BASE_SEED + run
    set_seed(seed)

    # Mouse split ONCE per run (joint-then-finetune uses mouse TRAIN only across all human folds)
    X_m_tr, X_m_va, y_m_tr, y_m_va = stratified_split_80_20(mouse_X, mouse_y_int, seed)
    X_m_tr, X_m_va, _ = standardize_train_only(X_m_tr, X_m_va, X_m_va.copy())[:3]

    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=seed)

    fold_base_accs, fold_base_f1s = [], []
    fold_jf_accs,    fold_jf_f1s    = [], []

    for fold_i, (trainval_idx, test_idx) in enumerate(skf.split(human_X, human_y_int), start=1):
        fold_seed = seed + 1000 * fold_i
        set_seed(fold_seed)

        trainval_idx = np.array(trainval_idx, dtype=int)
        test_idx     = np.array(test_idx,     dtype=int)

        # Inner split TRAIN/VAL from trainval pool
        tr_idx, va_idx = stratified_split_indices(trainval_idx, human_y_int, vf, fold_seed)

        # Pull fold data
        X_h_tr, y_h_tr = human_X[tr_idx],   human_y_int[tr_idx]
        X_h_va, y_h_va = human_X[va_idx],   human_y_int[va_idx]
        X_h_te, y_h_te = human_X[test_idx], human_y_int[test_idx]

        # Standardize HUMAN per fold (train-only)
        X_h_tr, X_h_va, X_h_te = standardize_train_only(X_h_tr, X_h_va, X_h_te)[:3]

        # --- Baseline
        base_acc, base_f1 = train_baseline_human_fold(
            run_idx=run+1, fold_idx=fold_i,
            X_tr=X_h_tr, y_tr=y_h_tr,
            X_va=X_h_va, y_va=y_h_va,
            X_te=X_h_te, y_te=y_h_te,
            num_classes=num_classes_coarse
        )

        # Reset seed so joint-then-finetune init is comparable with baseline within the fold
        set_seed(fold_seed)

        # --- Joint-then-finetune: joint MTL then human-only fine-tune (same split)
        jf_acc, jf_f1 = train_joint_finetune_fold(
            run_idx=run+1, fold_idx=fold_i,
            X_h_tr=X_h_tr, y_h_tr=y_h_tr,
            X_h_va=X_h_va, y_h_va=y_h_va,
            X_h_te=X_h_te, y_h_te=y_h_te,
            X_m_tr=X_m_tr, y_m_tr=y_m_tr,
            num_classes_h=num_classes_coarse, num_classes_m=num_classes_coarse
        )

        fold_base_accs.append(base_acc); fold_base_f1s.append(base_f1)
        fold_jf_accs.append(jf_acc);       fold_jf_f1s.append(jf_f1)

        print(f"[Run {run+1} Fold {fold_i}] BASE acc={base_acc:.4f} F1={base_f1:.4f}")
        print(f"[Run {run+1} Fold {fold_i}] JF   acc={jf_acc:.4f} F1={jf_f1:.4f}")
        print(f"[Run {run+1} Fold {fold_i}] Δacc={jf_acc-base_acc:+.4f} ΔF1={jf_f1-base_f1:+.4f}\n")

    # per-run = avg over folds
    base_acc_run = float(np.mean(fold_base_accs)); base_f1_run = float(np.mean(fold_base_f1s))
    jf_acc_run    = float(np.mean(fold_jf_accs));    jf_f1_run    = float(np.mean(fold_jf_f1s))

    run_base_accs.append(base_acc_run); run_base_f1s.append(base_f1_run)
    run_jf_accs.append(jf_acc_run);       run_jf_f1s.append(jf_f1_run)
    run_delta_accs.append(jf_acc_run - base_acc_run)
    run_delta_f1s.append(jf_f1_run  - base_f1_run)

    print(f"==================== RUN {run+1} (avg over {K_FOLDS} folds) ====================")
    print(f"BASE avg acc={base_acc_run:.4f} F1={base_f1_run:.4f}")
    print(f"JF   avg acc={jf_acc_run:.4f} F1={jf_f1_run:.4f}")
    print(f"Δ    avg acc={jf_acc_run-base_acc_run:+.4f} ΔF1={jf_f1_run-base_f1_run:+.4f}")
    print("===============================================================================\n")


# -----------------------------
# Summary over runs
# -----------------------------

run_base_accs  = np.array(run_base_accs,  dtype=float)
run_base_f1s   = np.array(run_base_f1s,   dtype=float)
run_jf_accs     = np.array(run_jf_accs,     dtype=float)
run_jf_f1s      = np.array(run_jf_f1s,      dtype=float)
run_delta_accs = np.array(run_delta_accs, dtype=float)
run_delta_f1s  = np.array(run_delta_f1s,  dtype=float)

def mean_sd(x):
    return float(x.mean()), float(x.std(ddof=1) if len(x) > 1 else 0.0)

m_ba, s_ba = mean_sd(run_base_accs);  m_bf, s_bf = mean_sd(run_base_f1s)
m_jfa, s_jfa = mean_sd(run_jf_accs);     m_jff, s_jff = mean_sd(run_jf_f1s)
m_da, s_da = mean_sd(run_delta_accs); m_df, s_df = mean_sd(run_delta_f1s)

print("=" * 80)
print(f"K-FOLD PAIRED RESULTS: K={K_FOLDS}, runs={N_RUNS}")
print(f"alpha_mouse={ALPHA_MOUSE} | LR_joint={LR_JOINT} | LR_finetune={LR_FT}")
print(f"BASE acc {m_ba:.4f} ± {s_ba:.4f} | F1 {m_bf:.4f} ± {s_bf:.4f}")
print(f"JF   acc {m_jfa:.4f} ± {s_jfa:.4f} | F1 {m_jff:.4f} ± {s_jff:.4f}")
print(f"Δacc  {m_da:+.4f} ± {s_da:.4f}")
print(f"ΔF1   {m_df:+.4f} ± {s_df:.4f}")
print("=" * 80)
